# Lesson 03
Run lessons 1–2 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第3课 多因子定价与选股

本课交付：FF5＋动量归因、真实 SEC 财务因子、公开时间审计、横截面去极值/标准化/行业与规模中性化、Rank IC、分组收益、单因子/等权/训练期加权对照、成本台账与验收。

**先运行第1、2课。** 本课共用第二课自融资多资产账本，所有方案统一成本和执行时间；教学数据窗为2014—2022，训练2014—2018、验证2019—2020、开发评估2021—2022。它们均属研究开发数据，不重新把已经看过的历史称为独立留出集。12只固定存续股票用于教学，存在存续与选择偏差；粗行业标签不是历史GICS。

风险因子：公开 FF5、动量组合用于解释已实现收益，不能当成当天可交易选股信号。选股因子：12−1月动量、年利润/最近资产（ROA）、账面权益/估计市值（B/M）；价值与质量来自实际报表。市值用最后已公开股数及其后拆股更新估计，无法反映所有回购/增发，须视为代理。

财务字段只在 filed 日纽约时间23:59以后可用，保守处理 companyfacts 只有日期的限制；按当时已公开版本选择最近报告期，记录 accession 与报告期，不用期末冒充公开日。拆股仅用于统一价格/股数单位；股息已经包含在复权收益中，不再重复加入现金。

方法顺序：03.01来源→03.02价格/报表→03.03因子面板→03.04归因→03.05 IC与组合→03.06诊断与验收。输出保留在本笔记本，缓存仅在内存，重启需重新下载。

来源：[SEC API](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、[SEC访问规则](https://www.sec.gov/about/developer-resources)、[French因子库](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)。French美国因子自2025年1月转用CRSP CIZ；此处保存当前版本哈希，回溯归因存在数据修订风险。


In [ ]:
# 03.01 共同环境与来源核验（03—05仅保存于Colab）
import sys, subprocess, importlib.metadata as im, time, requests, hashlib, io, zipfile, re, json
subprocess.check_call([sys.executable,'-m','pip','install','-q','yfinance==0.2.66','statsmodels==0.14.5','scikit-learn==1.6.1','lightgbm==4.6.0','beautifulsoup4==4.13.4'])
import numpy as np, pandas as pd, matplotlib.pyplot as plt, yfinance as yf
from bs4 import BeautifulSoup
from IPython.display import display, Markdown
CACHE=globals().get('CACHE',{}); SOURCE_LOG=globals().get('SOURCE_LOG',[])
SESSION=requests.Session(); SESSION.headers.update({'User-Agent':'QuantResearchTeachingNotebook/1.0 academic research','Accept-Encoding':'gzip, deflate'})
def fetch(url):
    if url in CACHE: return CACHE[url]
    if 'sec.gov' in url: time.sleep(0.7) # serial requests, below SEC 10/s limit
    r=SESSION.get(url,timeout=40)
    SOURCE_LOG.append({'url':url,'status':r.status_code,'bytes':len(r.content),'sha256':hashlib.sha256(r.content).hexdigest(),'retrieved_utc':pd.Timestamp.now(tz='UTC').isoformat()})
    r.raise_for_status(); CACHE[url]=r.content; return r.content
PROBE_URLS={
 'facts':'https://data.sec.gov/api/xbrl/companyfacts/CIK0000320193.json',
 'submissions':'https://data.sec.gov/submissions/CIK0000320193.json',
 'french':'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip',
 'LM':'https://drive.google.com/uc?export=download&id=1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF'}
for name,url in PROBE_URLS.items():
    if name=='LM': continue # full dictionary schema and fallback checked in 04.01
    try: print(name,'OK',len(fetch(url)))
    except Exception as e: print(name,type(e).__name__,str(e)[:160])
try:
    meta=json.loads(fetch(PROBE_URLS['submissions'])); q=pd.DataFrame(meta['filings']['recent']); row=q[q.form.eq('10-K')].iloc[0]
    ARCHIVE_PROBE='https://www.sec.gov/Archives/edgar/data/320193/'+row.accessionNumber.replace('-','')+'/'+row.primaryDocument
    print('filing',len(fetch(ARCHIVE_PROBE)))
except Exception as e: print('filing',type(e).__name__,str(e)[:160])
display(pd.DataFrame(SOURCE_LOG))
print('环境',sys.version.split()[0],{p:im.version(p) for p in ['numpy','pandas','yfinance','scikit-learn','lightgbm']})


In [ ]:
# 03.02 固定股票样本、复权价格与真实SEC报表缓存
from urllib.parse import urljoin
subprocess.check_call([sys.executable,'-m','pip','install','-q','pypdf==5.6.1'])
UNIVERSE={'AAPL':(320193,'tech'),'MSFT':(789019,'tech'),'IBM':(51143,'tech'),'INTC':(50863,'tech'),
 'KO':(21344,'consumer'),'PEP':(77476,'consumer'),'MCD':(63908,'consumer'),'PG':(80424,'consumer'),
 'XOM':(34088,'energy'),'CVX':(93410,'energy'),'CAT':(18230,'industrial'),'MMM':(66740,'industrial')}
ASSETS=list(UNIVERSE); NY='America/New_York'; SEED=42
START='2012-01-01'; END='2023-01-01'; TRAIN_END=pd.Timestamp('2019-01-01'); VALID_END=pd.Timestamp('2021-01-01')
raw=yf.download(ASSETS+['SPY'],start=START,end=END,auto_adjust=False,actions=True,progress=False,threads=False)
assert not raw.empty, 'Yahoo source empty: stop, do not fabricate prices'
CLOSE=raw['Adj Close'].reindex(columns=ASSETS+['SPY']).dropna(how='any')
QUOTE=raw['Close'].reindex(index=CLOSE.index,columns=CLOSE.columns)
OPEN=raw['Open'].reindex(index=CLOSE.index,columns=CLOSE.columns)*(CLOSE/QUOTE)
assert (OPEN>0).all().all() and (CLOSE>0).all().all()
SPLITS={}; FACTS={}
for t,(cik,industry) in UNIVERSE.items():
    sp=yf.Ticker(t).get_splits(period='max')
    sp.index=pd.DatetimeIndex(sp.index).tz_localize(None).normalize(); SPLITS[t]=sp[sp.ne(0)].copy()
    FACTS[t]=json.loads(fetch(f'https://data.sec.gov/api/xbrl/companyfacts/CIK{cik:010d}.json'))
PRICE_AUDIT={'provider':'Yahoo via yfinance','start':str(CLOSE.index.min()),'end':str(CLOSE.index.max()),
 'sha256':hashlib.sha256(pd.util.hash_pandas_object(raw,index=True).values.tobytes()).hexdigest(),
 'scope':'fixed survivors; adjusted synthetic units; no delisted universe','missing_removed':len(raw)-len(CLOSE)}
# 公司官网公开镜像：仅链接发现，不猜测/绕过SEC封禁。
IBM_IR='https://www.ibm.com/investor/services/investor-information'
IR_HTML=BeautifulSoup(fetch(IBM_IR),'html.parser')
IBM_LINKS=[{'label':a.get_text(' ',strip=True),'url':urljoin(IBM_IR,a['href'])} for a in IR_HTML.select('a[href]')
           if re.search(r'10.?[KQ]',a.get_text(' ',strip=True),re.I)]
print('价格审计',PRICE_AUDIT); display(pd.DataFrame(IBM_LINKS).tail(18))
# Google Drive公开下载端点；若返回HTML而非CSV，验收会失败并显示真实状态。
LM_URL='https://drive.usercontent.google.com/download?id=1iq2RUf8qGFEAk1g8wQntP3habOnR3fXF&export=download&authuser=0&confirm=t'
try:
    LM_BYTES=fetch(LM_URL); lm_probe=pd.read_csv(io.BytesIO(LM_BYTES)); assert {'Word','Negative','Positive'}.issubset(lm_probe.columns) and len(lm_probe)>10000,'Download returned HTML/quota page, not dictionary'; print('LM CSV schema verified',lm_probe.shape)
except Exception as e: LM_BYTES=None; print('LM_UNAVAILABLE',type(e).__name__,str(e)[:120])
print('03.02 数据读取结束；所有缓存仅在Colab内存')


In [ ]:
# 03.03 点时财务因子、拆股单位与行业/规模中性化
TAGS={'assets':('us-gaap',['Assets'],'USD'),
 'equity':('us-gaap',['StockholdersEquity','StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest'],'USD'),
 'income':('us-gaap',['NetIncomeLoss','ProfitLoss'],'USD'),
 'shares':('dei',['EntityCommonStockSharesOutstanding'],'shares')}
FACT_AUDIT=[]
def fact_events(t,key):
    ns,tags,unit=TAGS[key]; records=[]
    for priority,tag in enumerate(tags):
        for x in FACTS[t]['facts'].get(ns,{}).get(tag,{}).get('units',{}).get(unit,[]):
            if x.get('form') not in ['10-K','10-Q','10-K/A','10-Q/A']: continue
            if key=='income':
                if 'start' not in x or not 330<=(pd.Timestamp(x['end'])-pd.Timestamp(x['start'])).days<=380: continue
            records.append(dict(x,tag=tag,priority=priority))
    assert records, (t,key,'no usable SEC tag')
    f=pd.DataFrame(records); f['end']=pd.to_datetime(f.end); f['filed']=pd.to_datetime(f.filed)
    f=f.sort_values(['filed','end','priority','accn'],ascending=[True,True,False,True])
    current=None; updates=[]
    for _,r in f.iterrows():
        if current is None or r.end>current.end or (r.end==current.end and r.filed>=current.filed): current=r
        else: continue # later comparative old periods do not replace the latest known period
        updates.append({'available_at':(r.filed+pd.Timedelta(hours=23,minutes=59)).tz_localize(NY),
                        key:float(r.val),key+'_end':r.end,key+'_accn':r.accn,key+'_tag':r.tag})
    return pd.DataFrame(updates).drop_duplicates('available_at',keep='last').sort_values('available_at')
FUND={}; dates=CLOSE.index
for t in ASSETS:
    p=pd.DataFrame({'date':dates,'decision_at':(dates+pd.Timedelta(hours=16)).tz_localize(NY)})
    for key in TAGS:
        ev=fact_events(t,key).rename(columns={'available_at':key+'_available_at'})
        p=pd.merge_asof(p.sort_values('decision_at'),ev,left_on='decision_at',right_on=key+'_available_at',direction='backward')
    p=p.set_index('date'); sp=SPLITS[t]
    # Yahoo Close统一到当前拆股股数基准；还原当日名义报价后与当时公开股数匹配。
    future_split=pd.Series([sp.loc[sp.index>d].prod() for d in dates],index=dates)
    nominal_close=QUOTE[t]*future_split
    share_update=pd.Series([sp.loc[(sp.index>e)&(sp.index<=d)].prod() if pd.notna(e) else np.nan for d,e in zip(dates,p.shares_end)],index=dates)
    p['marketcap_est']=nominal_close*p.shares*share_update
    p['value']=p.equity/p.marketcap_est.where(p.marketcap_est>0)
    p['quality']=p.income/p.assets.where(p.assets>0)
    p['momentum']=CLOSE[t].shift(21)/CLOSE[t].shift(252)-1
    p['low_vol']=-CLOSE[t].pct_change(fill_method=None).rolling(63).std()
    p['size']=np.log(p.marketcap_est.where(p.marketcap_est>0))
    for key in TAGS:
        p[key+'_age_days']=(pd.Series(dates,index=dates)-p[key+'_end']).dt.days
        assert (p[key+'_available_at'].dropna()<=p.loc[p[key+'_available_at'].notna(),'decision_at']).all()
    stale=p[[k+'_age_days' for k in TAGS]].max(axis=1)>550
    p.loc[stale,['quality','value','size']]=np.nan
    p['ticker']=t; p['industry']=UNIVERSE[t][1]; FUND[t]=p
PANEL=pd.concat(FUND.values()).reset_index().sort_values(['date','ticker']).reset_index(drop=True)
PANEL=PANEL[PANEL.date.between('2014-01-01','2022-12-31')].copy()
FACTOR_NAMES=['momentum','quality','value']
def neutralize(g,col):
    ok=g[col].notna()&g['size'].notna(); out=pd.Series(np.nan,index=g.index)
    if ok.sum()<8: return out
    y=g.loc[ok,col].clip(g.loc[ok,col].quantile(.05),g.loc[ok,col].quantile(.95))
    y=(y-y.mean())/max(y.std(ddof=0),1e-12)
    industry=pd.get_dummies(g.loc[ok,'industry'],dtype=float)
    sz=g.loc[ok,'size']; sz=(sz-sz.mean())/max(sz.std(ddof=0),1e-12)
    X=np.column_stack([industry.to_numpy(),sz.to_numpy()]); residual=y.to_numpy()-X@np.linalg.lstsq(X,y,rcond=None)[0]
    out.loc[ok]=residual/max(residual.std(),1e-12); return out
for col in FACTOR_NAMES:
    PANEL[col+'_z']=pd.concat([neutralize(g,col) for _,g in PANEL.groupby('date',sort=False)]).reindex(PANEL.index)
MONTH_DATES=pd.DatetimeIndex(PANEL.groupby(PANEL.date.dt.to_period('M')).date.max().values)
MONTH=PANEL[PANEL.date.isin(MONTH_DATES)].dropna(subset=[f+'_z' for f in FACTOR_NAMES]).copy()
L3_DICTIONARY=pd.DataFrame([
 ['momentum','adjusted close[t-21]/close[t-252]-1','t close','skip latest month'],
 ['quality','annual net income/latest assets','filed day 23:59 NY','ROA proxy, mixed balance sheet date'],
 ['value','equity/estimated nominal marketcap','filed day 23:59 NY','lagged disclosed shares; split correction'],
 ['neutralization','5/95% clip -> z -> industry/size OLS residual','same date cross-section','fixed study industry tags, small universe']],columns=['feature','definition','available','limitation'])
display(L3_DICTIONARY); display(MONTH[['date','ticker','momentum','quality','value','shares_end','income_available_at','equity_accn']].tail(12))
print('03.03 PASS',len(PANEL),'daily rows',len(MONTH),'monthly rows; missing rates'); display(PANEL[FACTOR_NAMES+['size']].isna().mean())


In [ ]:
# 03.04 French研究因子、FF3/FF5+动量与共同策略归因
import statsmodels.api as sm
def french_monthly(url):
    z=zipfile.ZipFile(io.BytesIO(fetch(url))); txt=z.read(z.namelist()[0]).decode('utf-8-sig')
    lines=txt.splitlines(); start=next(i for i,x in enumerate(lines) if re.match(r'^\s*\d{6},',x)); head=lines[start-1]
    rows=[]
    for x in lines[start:]:
        if not re.match(r'^\s*\d{6},',x): break
        rows.append(x)
    f=pd.read_csv(io.StringIO(head+'\n'+'\n'.join(rows)),index_col=0)
    f.columns=f.columns.str.strip(); f.index=pd.PeriodIndex(f.index.astype(str),freq='M')
    return f.replace([-99.99,-999],np.nan)/100
FF5=french_monthly(PROBE_URLS['french'])
MOM=french_monthly('https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip')
FF=FF5.join(MOM.rename(columns={MOM.columns[0]:'Mom'}),how='inner')
# 第一课风格的SPY趋势策略重建。归因用close-to-close收益，与月度French口径对齐。
trend=(CLOSE.SPY.rolling(50).mean()>CLOSE.SPY.rolling(200).mean()).astype(float)
close_ret=CLOSE.SPY.pct_change(fill_method=None)
strategy_daily=.8*trend.shift(1).fillna(0)*close_ret # descriptive frictionless close-based research proxy
strategy_month=(1+strategy_daily).groupby(strategy_daily.index.to_period('M')).prod()-1
research=pd.concat([strategy_month.rename('strategy'),FF],axis=1).dropna().loc['2014-01':'2022-12']
ATTR={}; attr_rows=[]
for name,cols in {'FF3':['Mkt-RF','SMB','HML'],'FF5':['Mkt-RF','SMB','HML','RMW','CMA'],'FF5+Mom':['Mkt-RF','SMB','HML','RMW','CMA','Mom']}.items():
    m=sm.OLS(research.strategy-research.RF,sm.add_constant(research[cols])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
    ATTR[name]=m
    for term in m.params.index: attr_rows.append({'model':name,'term':term,'coefficient':m.params[term],'HAC_t':m.tvalues[term],'R2':m.rsquared})
display(pd.DataFrame(attr_rows)); print('月alpha为回归截距，不是alpha×12保证收益；该归因为无成本close代理，不混入open账本。')
fig,ax=plt.subplots(1,2,figsize=(13,4)); research[['strategy','Mkt-RF','HML','RMW','Mom']].cumsum().plot(ax=ax[0],title='Monthly arithmetic cumulative factor diagnostics')
FF.loc['2014-01':'2022-12',['Mkt-RF','SMB','HML','RMW','CMA','Mom']].corr().plot.bar(ax=ax[1],title='Published factor correlations'); plt.tight_layout(); plt.show()
assert len(research)==108 and np.isfinite(research.to_numpy()).all()
print('03.04 PASS: FF3, FF5, FF5+Mom; percent converted once; 108 aligned months')


In [ ]:
# 03.05 Rank IC、训练期加权、分组与统一成本组合
assert 'l2_simulate_multi' in globals(),'请先顺序运行第1、2课'
def forward_open(h): return OPEN[ASSETS].shift(-(h+1))/OPEN[ASSETS].shift(-1)-1
def attach_forward(df,h):
    y=forward_open(h).stack().rename('forward_return').rename_axis(['date','ticker']).reset_index()
    out=df.merge(y,on=['date','ticker'],how='left'); out['label_end']=out.date.map(pd.Series(CLOSE.index,index=CLOSE.index).shift(-(h+1)))
    return out
M21=attach_forward(MONTH,21).dropna(subset=['forward_return'])
IC_ROWS=[]
for date,g in M21.groupby('date'):
    for f in FACTOR_NAMES: IC_ROWS.append({'date':date,'factor':f,'RankIC':g[f+'_z'].corr(g.forward_return,method='spearman'),'raw_RankIC':g[f].corr(g.forward_return,method='spearman'),'n':len(g),'label_end':g.label_end.max()})
IC=pd.DataFrame(IC_ROWS)
IC['fold']=np.select([IC.date<TRAIN_END,IC.date<VALID_END],['train','validation'],default='development_eval')
print('同一日期/同一股票集合：中性化前后Rank IC对照')
display(IC.groupby(['fold','factor'])[['raw_RankIC','RankIC']].mean())
train_ic=IC[(IC.date<TRAIN_END)&(IC.label_end<TRAIN_END)].groupby('factor').RankIC.mean().reindex(FACTOR_NAMES)
weights=train_ic.clip(lower=0)
FACTOR_WEIGHTS=weights/weights.sum() if weights.sum()>0 else pd.Series(1/3,index=FACTOR_NAMES)
MONTH['equal']=MONTH[[f+'_z' for f in FACTOR_NAMES]].mean(axis=1)
MONTH['trained']=sum(MONTH[f+'_z']*FACTOR_WEIGHTS[f] for f in FACTOR_NAMES)
SCORE_COLUMNS={f:f+'_z' for f in FACTOR_NAMES}|{'equal':'equal','trained':'trained'}
exec_dates=OPEN.loc['2014-02-01':'2022-12-29'].index
L3_TARGETS={}; L3_LEDGERS={}; L3_TRADES={}
def monthly_target(score_col,quantile=None):
    decisions=pd.DataFrame(0.,index=MONTH_DATES,columns=ASSETS)
    for date,g in MONTH.groupby('date'):
        ranked=g.sort_values([score_col,'ticker']); chosen=ranked.tail(3)
        if quantile is not None: chosen=np.array_split(ranked,3)[quantile]
        if len(chosen): decisions.loc[date,chosen.ticker]=.8/len(chosen)
    # 收盘信号只能从下个交易日开盘启用；月内每日维护同一目标权重，费用覆盖维护交易。
    sparse=decisions.reindex(OPEN.index)
    return sparse.shift(1).ffill().fillna(0).reindex(exec_dates)
for name,col in SCORE_COLUMNS.items():
    target=monthly_target(col); L3_TARGETS[name]=target
    ledger,trades=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
    L3_LEDGERS[name]=ledger; L3_TRADES[name]=trades
for k in range(3):
    target=monthly_target('equal',k); name='Q'+str(k+1); L3_TARGETS[name]=target
    L3_LEDGERS[name],L3_TRADES[name]=l2_simulate_multi(OPEN[ASSETS],target,commission=3,slippage=2,borrow_rate=0,funding=0,initial=10000)
def ledger_stats(x):
    growth=(1+x.net_return).cumprod(); dd=growth/growth.cummax().clip(lower=1)-1
    return {'net_return':growth.iloc[-1]-1,'annualized':growth.iloc[-1]**(252/len(x))-1,'max_drawdown':dd.min(),
            'Sharpe_0rf':x.net_return.mean()/max(x.net_return.std(),1e-12)*np.sqrt(252),
            'turnover_sum':x.turnover.sum(),'fees':x.transaction_cost.sum()}
L3_RESULT=pd.DataFrame([{'model':n,'fold':fold,**ledger_stats(x.loc[a:b])} for n,x in L3_LEDGERS.items()
    for fold,a,b in [('train','2014-02-01','2018-12-31'),('validation','2019-01-01','2020-12-31'),('development_eval','2021-01-01','2022-12-29')]]).set_index(['fold','model'])
display(pd.DataFrame({'train_mean_IC':train_ic,'frozen_weight':FACTOR_WEIGHTS})); display(L3_RESULT)
print('权重仅训练期估计；验证与开发评估报告全部对照，未按最后一段结果挑选赢家。')
fig,ax=plt.subplots(1,2,figsize=(13,4)); pd.DataFrame({n:x.wealth for n,x in L3_LEDGERS.items() if not n.startswith('Q')}).plot(ax=ax[0],title='Same cost, daily target maintenance')
pd.DataFrame({n:x.wealth for n,x in L3_LEDGERS.items() if n.startswith('Q')}).plot(ax=ax[1],title='Tercile portfolios, net of costs'); plt.tight_layout(); plt.show()
print('03.05 PASS',len(IC),'IC observations;',len(L3_LEDGERS),'shared-ledger portfolios')


In [ ]:
# 03.06 IC衰减、相关性、行业敞口与因果性验收
DECAY=[]
for h in [5,21,63]:
    joined=attach_forward(MONTH,h).dropna(subset=['forward_return'])
    for f in FACTOR_NAMES:
        vals=[g[f+'_z'].corr(g.forward_return,method='spearman') for _,g in joined.groupby('date')]
        DECAY.append({'horizon_sessions':h,'factor':f,'mean_RankIC':np.nanmean(vals),'months':len(vals)})
display(pd.DataFrame(DECAY)); display(IC.groupby('factor').RankIC.agg(['mean','std','count']))
corr=MONTH[[f+'_z' for f in FACTOR_NAMES]].corr(); display(corr)
industry_weights=pd.DataFrame({group:L3_TARGETS['equal'][[t for t in ASSETS if UNIVERSE[t][1]==group]].sum(axis=1) for group in sorted({v[1] for v in UNIVERSE.values()})})
display(industry_weights.mean().rename('average_target_weight').to_frame())
fig,ax=plt.subplots(2,2,figsize=(14,8)); IC.pivot(index='date',columns='factor',values='RankIC').rolling(12,min_periods=3).mean().plot(ax=ax[0,0],title='12-month rolling mean Rank IC')
for f,g in pd.DataFrame(DECAY).groupby('factor'): ax[0,1].plot(g.horizon_sessions,g.mean_RankIC,marker='o',label=f)
ax[0,1].legend(); ax[0,1].set_title('Horizon decay diagnostic')
industry_weights.plot.area(ax=ax[1,0],title='Study industry exposure'); L3_LEDGERS['equal'].turnover.rolling(21).sum().plot(ax=ax[1,1],title='21-session turnover including maintenance'); plt.tight_layout(); plt.show()
CHECKS=[]
def check(name,ok):
    assert bool(ok),name
    CHECKS.append({'check':name,'status':'PASS'})
check('SEC PIT availability audit',all((PANEL[k+'_available_at'].dropna()<=PANEL.loc[PANEL[k+'_available_at'].notna(),'decision_at']).all() for k in TAGS))
check('training IC labels matured before validation',IC[(IC.date<TRAIN_END)&(IC.label_end<TRAIN_END)].label_end.max()<TRAIN_END)
check('daily asset-date unique',not PANEL.duplicated(['date','ticker']).any())
check('French six factors and RF',set(['Mkt-RF','SMB','HML','RMW','CMA','Mom','RF']).issubset(FF.columns))
# 中性化的行业残差均值应近零；规模残差正交。
for f in FACTOR_NAMES:
    valid=PANEL.dropna(subset=[f+'_z','size'])
    check(f+' industry orthogonality',valid.groupby(['date','industry'])[f+'_z'].mean().abs().max()<1e-7)
    dots=valid.groupby('date').apply(lambda g: np.dot(g[f+'_z'],g['size']-g['size'].mean()),include_groups=False)
    check(f+' size orthogonality',dots.abs().max()<1e-6)
for name,x in L3_LEDGERS.items():
    check(name+' common window',x.index.equals(exec_dates))
    check(name+' equity conservation',np.allclose(x.net_pnl,x.market_pnl-x.transaction_cost))
    check(name+' long-only and gross cap',(L3_TARGETS[name]>=0).all().all() and (x.gross_exposure<=.800001).all())
    check(name+' compounded wealth',np.allclose(x.wealth,(1+x.net_return).cumprod()))
# 有效报表公开时间与信号时间各自保存；12-1动量不能读当日未来价格。
probe_date=MONTH.date.iloc[len(MONTH)//2]; probe_t=MONTH.ticker.iloc[len(MONTH)//2]
past=CLOSE[probe_t].loc[:probe_date]; expected=past.iloc[-22]/past.iloc[-253]-1
check('momentum prefix recomputation',np.isclose(MONTH.loc[(MONTH.date==probe_date)&(MONTH.ticker==probe_t),'momentum'].iloc[0],expected))
# 对03.05同一有成本账户做close-to-close因子归因，避免把开盘收益直接对上收盘因子。
net_account=L3_LEDGERS['equal']
account_close=net_account.cash+sum(net_account['units_'+a]*CLOSE[a].reindex(net_account.index) for a in ASSETS)
month_equity=account_close.groupby(account_close.index.to_period('M')).last()
month_equity.loc[pd.Period('2014-01',freq='M')]=10000.; month_equity=month_equity.sort_index()
actual_month=month_equity.pct_change(fill_method=None).rename('account_net')
# 最后2022-12仅到29日收盘，排除这个不完整月份。
actual_factor_data=pd.concat([actual_month,FF],axis=1).loc['2014-02':'2022-11'].dropna()
ACTUAL_ATTR=sm.OLS(actual_factor_data.account_net-actual_factor_data.RF,
                  sm.add_constant(actual_factor_data[['Mkt-RF','SMB','HML','RMW','CMA','Mom']])).fit(cov_type='HAC',cov_kwds={'maxlags':3})
print('同一个等权因子有成本账户：完整月份收盘权益，FF5+Mom归因')
display(pd.DataFrame({'coefficient':ACTUAL_ATTR.params,'HAC_t':ACTUAL_ATTR.tvalues,'p_value':ACTUAL_ATTR.pvalues}))
check('actual portfolio FF5+Mom aligns full months',len(actual_factor_data)==106 and actual_factor_data.index.max()==pd.Period('2022-11',freq='M'))
L3_VALIDATION=pd.DataFrame(CHECKS); display(L3_VALIDATION)
L3_CARD={'course':3,'status':'verified','data_card':PRICE_AUDIT,'weights':FACTOR_WEIGHTS.to_dict(),'checks':len(CHECKS),
 'limitations':['12 fixed survivors','coarse static industry','marketcap uses lagged shares','SEC revised taxonomy/vendor revised prices','daily target maintenance, not literal monthly buy-and-hold','HAC descriptive inference; separate close proxy and same-cost account attribution','no pristine holdout claim']}
print(json.dumps(L3_CARD,ensure_ascii=False,indent=2)); print('L3 FINAL PASS')
